# Week 5 Exercise — AI Knowledge RAG Assistant

This project implements a simple Retrieval-Augmented Generation (RAG) system that answers
questions about AI concepts such as RAG, embeddings, transformers, and vector databases.

Documents are loaded from a small knowledge base, split into chunks, converted into vector
embeddings, and stored in a Chroma vector database. When a user asks a question, the system
retrieves the most relevant document chunks and provides them as context to a language model
to generate an accurate response.

The application uses LangChain for the RAG pipeline, HuggingFace embeddings for semantic
search, and Gradio to provide an interactive chat interface.


In [ ]:
import os
from dotenv import load_dotenv

import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma

from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()


In [ ]:
MODEL = "gpt-4o-mini"
DOCS_PATH = "docs"
DB_NAME = "ai_knowledge_vector_db"

print("Model:", MODEL)
print("Documents:", DOCS_PATH)

In [ ]:
loader = DirectoryLoader(
    DOCS_PATH,
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(doc.metadata)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

print("\nExample chunk:\n")
print(chunks[0].page_content[:300])

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-l6-v2"
)

# test embedding
test_vec = embeddings.embed_query("What is RAG?")
print("Embedding length:", len(test_vec))

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ai_knowledge_docs",
    persist_directory=DB_NAME
)

print("Vector database created.")
print("Number of vectors:", vectorstore._collection.count())

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k":8})

# quick test retrieval
query = "What is RAG?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs,1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:200])

In [ ]:
llm = ChatOpenAI(
    model=MODEL,
    temperature=0
)

In [ ]:
SYSTEM_PROMPT = """
You are an AI assistant that answers questions about AI concepts.

Use the provided context to answer the user's question.
If the answer is not in the context, say you don't know.

Context:
{context}
"""

def answer_question(question, history):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    system_prompt = SYSTEM_PROMPT.format(context=context)

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])

    return response.content


In [ ]:
print(answer_question("What is Retrieval Augmented Generation?", []))

In [ ]:
with gr.Blocks(title="AI Knowledge RAG Assistant") as ui:

    gr.Markdown("# AI Knowledge RAG Assistant")
    gr.Markdown("Ask questions about AI concepts like RAG, embeddings, transformers, and vector databases.")

    chatbot = gr.ChatInterface(
        fn=answer_question
    )

ui.launch()